In [12]:
import math
import time
import numpy as np
import torch

# ============================================================
# Restricted Coercivity Scanner (SU(2), 4D torus)
#
# Goal:
#   Empirically test the "restricted quartic coercivity" candidate on K_Λ(r):
#
#     P_Λ(U)  :=  Σ_p z_p(U) <∇S_W(U), ∇z_p(U)>
#     Z2_Λ(U) :=  Σ_p z_p(U)^2
#
#   We estimate P_Λ(U) by Monte Carlo Gaussian directions + symmetric finite differences:
#     Γ(S, z_p) = E[ (D_X S)(D_X z_p) ]  for X ~ N(0, I)
#   so
#     P_Λ(U) = E[ (D_X S) * Σ_p z_p(U) (D_X z_p) ].
#
#   Then we scan ratios on the small-field set K_Λ(r):
#     ratio(U)     = P_hat / Z2
#     ratio_LCB(U) = (P_hat - nsigma*SE) / Z2   (conservative, avoids MC false negatives)
#
# Interpretation:
#   - If min ratio_LCB over {theta_max <= r} is > 0, this supports:
#         P_Λ(U) >= c(r) * Σ z_p(U)^2   on K_Λ(r)
#   - If you find ratio_LCB <= 0 for some U in K_Λ(r), you found a counterexample (at that r).
#
# Notes:
#   - "r" is measured in SU(2) rotation angle θ, where a quaternion has scalar part a=cos θ.
#   - Uses float64 for U and defects; Xi directions are float32 for memory.
#   - Designed to run on A100 by increasing L, K_total, mc.
# ============================================================

# --------------------------
# USER CONFIG (tune freely)
# --------------------------
CFG = dict(
    device="auto",     # "auto" or "cuda" or "cpu"
    L=8,               # even. Try 8 or 12 on A100; 16 is heavy.
    beta=6.0,

    # small-field sampling amplitudes (U_b = exp(sigma * N(0,1)) per link)
    sigmas=[0.02, 0.04, 0.06, 0.08],

    # small-field thresholds r (in radians, SU(2) angle arccos(a_p))
    r_list=[0.25, 0.35, 0.45, 0.60],

    # number of configurations per sigma
    K_total=256,
    batch_cfg=8,       # configs per batch (memory knob)

    # finite-difference + MC
    eps_fd=1e-4,
    mc=256,            # directions per config
    mc_chunk=8,        # MC chunk (memory knob)

    # conservative lower confidence bound
    nsigma=2.0,

    # avoid divide by tiny Z2
    z2_min=1e-14,

    # seeds + output
    seed_base=1234,
    out_npz="restricted_coercivity_scan.npz",
)

# ==========================
# SU(2) quaternion utilities
# ==========================
def su2_mul(q1, q2):
    a1, b1, c1, d1 = q1.unbind(-1)
    a2, b2, c2, d2 = q2.unbind(-1)
    return torch.stack(
        [
            a1 * a2 - b1 * b2 - c1 * c2 - d1 * d2,
            a1 * b2 + b1 * a2 + c1 * d2 - d1 * c2,
            a1 * c2 - b1 * d2 + c1 * a2 + d1 * b2,
            a1 * d2 + b1 * c2 - c1 * b2 + d1 * a2,
        ],
        dim=-1,
    )

def su2_conj(q):
    a, b, c, d = q.unbind(-1)
    return torch.stack([a, -b, -c, -d], dim=-1)

def su2_exp(v):
    # v (...,3) -> quaternion (...,4)
    theta = torch.linalg.norm(v, dim=-1, keepdim=True)
    a = torch.cos(theta)
    theta2 = theta * theta
    s_over = torch.where(
        theta > 1e-8,
        torch.sin(theta) / theta,
        1.0 - theta2 / 6.0 + (theta2 * theta2) / 120.0,
    )
    vec = s_over * v
    return torch.cat([a, vec], dim=-1)

# ==========================
# Lattice ops (4D torus)
# U shape: (B, L,L,L,L, 4, 4)  [mu, quaternion]
# ==========================
def roll4(x, mu, shift):
    # x shape (..., L,L,L,L, 4) with last dim quaternion
    start = x.ndim - 1 - 4
    return torch.roll(x, shifts=shift, dims=start + mu)

def plaquette(U, mu, nu):
    U_mu = U[..., mu, :]
    U_nu = U[..., nu, :]
    U_nu_xpmu = roll4(U_nu, mu, -1)
    U_mu_xpnu = roll4(U_mu, nu, -1)
    return su2_mul(
        su2_mul(
            su2_mul(U_mu, U_nu_xpmu),
            su2_conj(U_mu_xpnu),
        ),
        su2_conj(U_nu),
    )

def all_defects(U):
    # returns stacked defects z_p = 1 - q0(P) for all 6 planes
    zs = []
    for mu in range(4):
        for nu in range(mu + 1, 4):
            P = plaquette(U, mu, nu)
            zs.append(1.0 - P[..., 0])
    return torch.stack(zs, dim=0)  # (6, ..., L,L,L,L)

# ==========================
# Sampling configs: U_b = exp(sigma * N(0,1)) per link
# ==========================
@torch.no_grad()
def sample_configs(L, B, sigma, device, dtype=torch.float64, xi_dtype=torch.float32, seed=0):
    g = torch.Generator(device=device)
    g.manual_seed(int(seed))
    Xi = torch.randn((B, L, L, L, L, 4, 3), device=device, dtype=xi_dtype, generator=g)
    return su2_exp((sigma * Xi).to(dtype))

# ==========================
# Estimate P and compute Z2 + theta_max per config
# ==========================
@torch.no_grad()
def estimate_P_z2_tmax(U, beta=6.0, eps_fd=1e-4, mc=256, mc_chunk=8, seed=0, xi_dtype=torch.float32):
    device = U.device
    dtype = U.dtype
    B = U.shape[0]

    # Base defects
    z0 = all_defects(U)  # (6,B,L,L,L,L)
    z0flat = z0.reshape(6, B, -1).permute(1, 0, 2).reshape(B, -1)  # (B, Np)
    z2 = (z0flat * z0flat).sum(dim=1)                               # (B,)
    zmax = z0flat.max(dim=1).values                                 # (B,)

    # Max plaquette angle theta_max (SU(2): scalar part a_p = cos theta)
    p0 = 1.0 - z0
    theta = torch.acos(torch.clamp(p0, -1.0, 1.0))
    theta_max = theta.reshape(6, B, -1).amax(dim=2).amax(dim=0)      # (B,)

    # Monte Carlo for P per config
    g = torch.Generator(device=device)
    g.manual_seed(int(seed))

    sumP = torch.zeros((B,), device=device, dtype=torch.float64)
    sumP2 = torch.zeros((B,), device=device, dtype=torch.float64)

    done = 0
    U0 = U.unsqueeze(0)  # (1,B,...)
    while done < mc:
        k = min(mc_chunk, mc - done)

        Xi = torch.randn((k, *U.shape[:5], 4, 3), device=device, dtype=xi_dtype, generator=g)
        exp_p = su2_exp((eps_fd * Xi).to(dtype))
        exp_m = su2_exp((-eps_fd * Xi).to(dtype))

        Up = su2_mul(U0, exp_p)
        Um = su2_mul(U0, exp_m)

        zp = all_defects(Up)  # (6,k,B,...)
        zm = all_defects(Um)

        # flatten to (k,B,Np)
        zp_flat = zp.reshape(6, k, B, -1).permute(1, 2, 0, 3).reshape(k, B, -1)
        zm_flat = zm.reshape(6, k, B, -1).permute(1, 2, 0, 3).reshape(k, B, -1)

        dz = (zp_flat - zm_flat) / (2.0 * eps_fd)  # D_X z_p
        dS = beta * (zp_flat.sum(dim=2) - zm_flat.sum(dim=2)) / (2.0 * eps_fd)  # D_X S

        # P sample: (D_X S) * Σ_p z_p(U) (D_X z_p)
        inner = (dz * z0flat.unsqueeze(0)).sum(dim=2)  # (k,B)
        Pk = dS * inner                                # (k,B)

        sumP += Pk.sum(dim=0).to(torch.float64)
        sumP2 += (Pk * Pk).sum(dim=0).to(torch.float64)

        done += k
        del Xi, exp_p, exp_m, Up, Um, zp, zm, zp_flat, zm_flat, dz, dS, inner, Pk
        if device.type == "cuda":
            torch.cuda.empty_cache()

    P_hat = sumP / float(mc)
    var = (sumP2 - (sumP * sumP) / float(mc)) / float(max(1, mc - 1))
    var = torch.clamp(var, min=0.0)
    P_se = torch.sqrt(var / float(mc))
    return P_hat, P_se, z2, zmax, theta_max

# ==========================
# Main scan
# ==========================
def main():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu") if CFG["device"] == "auto" else torch.device(CFG["device"])
    print(f"[device] {device} {torch.cuda.get_device_name(0) if device.type=='cuda' else ''}")

    L = int(CFG["L"])
    assert L % 2 == 0, "Use even L for a periodic 4-torus test."
    beta = float(CFG["beta"])
    eps_fd = float(CFG["eps_fd"])
    mc = int(CFG["mc"])
    mc_chunk = int(CFG["mc_chunk"])
    batch_cfg = int(CFG["batch_cfg"])
    nsigma = float(CFG["nsigma"])
    z2_min = float(CFG["z2_min"])
    sigmas = list(map(float, CFG["sigmas"]))
    r_list = list(map(float, CFG["r_list"]))
    K_total = int(CFG["K_total"])
    seed_base = int(CFG["seed_base"])
    out_npz = CFG.get("out_npz", None)

    t0 = time.time()

    npz_dict = {"CFG_text": str(CFG)}

    for sigma in sigmas:
        P_all, Pse_all, z2_all, zmax_all, tmax_all, seed_all = [], [], [], [], [], []
        n_done = 0

        while n_done < K_total:
            B = min(batch_cfg, K_total - n_done)
            seed_cfg = seed_base + 100000 * int(round(1000 * sigma)) + n_done

            U = sample_configs(L, B, sigma, device=device, dtype=torch.float64, xi_dtype=torch.float32, seed=seed_cfg)
            P_hat, P_se, z2, zmax, tmax = estimate_P_z2_tmax(
                U, beta=beta, eps_fd=eps_fd, mc=mc, mc_chunk=mc_chunk, seed=seed_cfg + 7, xi_dtype=torch.float32
            )

            P_all.append(P_hat.detach().cpu())
            Pse_all.append(P_se.detach().cpu())
            z2_all.append(z2.detach().cpu())
            zmax_all.append(zmax.detach().cpu())
            tmax_all.append(tmax.detach().cpu())
            seed_all.append(torch.full((B,), seed_cfg, dtype=torch.int64))

            n_done += B
            del U, P_hat, P_se, z2, zmax, tmax
            if device.type == "cuda":
                torch.cuda.empty_cache()

        P = torch.cat(P_all)
        Pse = torch.cat(Pse_all)
        z2 = torch.cat(z2_all)
        zmax = torch.cat(zmax_all)
        tmax = torch.cat(tmax_all)
        seeds = torch.cat(seed_all)

        ratio = P / torch.clamp(z2, min=z2_min)
        ratio_lcb = (P - nsigma * Pse) / torch.clamp(z2, min=z2_min)

        print(f"\n=== sigma={sigma}  (N={K_total}, beta={beta}, mc={mc}) ===")
        print(f"theta_max: min={float(tmax.min()):.6g}  med={float(tmax.median()):.6g}  max={float(tmax.max()):.6g}")
        print(f"zmax:      min={float(zmax.min()):.6g}  med={float(zmax.median()):.6g}  max={float(zmax.max()):.6g}")

        for r in r_list:
            mask = (tmax <= r) & (z2 > z2_min)
            cnt = int(mask.sum().item())
            if cnt == 0:
                print(f" r={r:.6g}: cnt=0")
                continue

            rr = ratio[mask]
            rl = ratio_lcb[mask]

            # identify the worst-LCB config and report its seed for reproducibility
            idxs = torch.nonzero(mask, as_tuple=False).flatten()
            worst_local = int(torch.argmin(rl).item())
            worst_idx = int(idxs[worst_local].item())

            print(
                f" r={r:.6g}: cnt={cnt:5d}  "
                f"min_ratio={float(rr.min()):.6g}  min_ratio_LCB={float(rl.min()):.6g}  "
                f"p05={float(rr.quantile(0.05)):.6g}  med={float(rr.median()):.6g}  "
                f"worst_seed={int(seeds[worst_idx].item())}"
            )

        tag = f"sigma_{sigma:.6g}".replace(".", "p")
        npz_dict[f"{tag}_P"] = P.numpy()
        npz_dict[f"{tag}_Pse"] = Pse.numpy()
        npz_dict[f"{tag}_z2"] = z2.numpy()
        npz_dict[f"{tag}_zmax"] = zmax.numpy()
        npz_dict[f"{tag}_tmax"] = tmax.numpy()
        npz_dict[f"{tag}_seed"] = seeds.numpy()

    dt = time.time() - t0
    print(f"\n[done] wall={dt/60:.2f} min")

    if out_npz:
        np.savez_compressed(out_npz, **npz_dict)
        print(f"[saved] {out_npz}")

if __name__ == "__main__":
    main()


[device] cuda NVIDIA A100-SXM4-40GB

=== sigma=0.02  (N=256, beta=6.0, mc=256) ===
theta_max: min=0.174621  med=0.193814  max=0.247499
zmax:      min=0.0152075  med=0.0187232  max=0.0304718
 r=0.25: cnt=  256  min_ratio=84.9633  min_ratio_LCB=68.6372  p05=91.5439  med=107.438  worst_seed=2001458
 r=0.35: cnt=  256  min_ratio=84.9633  min_ratio_LCB=68.6372  p05=91.5439  med=107.438  worst_seed=2001458
 r=0.45: cnt=  256  min_ratio=84.9633  min_ratio_LCB=68.6372  p05=91.5439  med=107.438  worst_seed=2001458
 r=0.6: cnt=  256  min_ratio=84.9633  min_ratio_LCB=68.6372  p05=91.5439  med=107.438  worst_seed=2001458

=== sigma=0.04  (N=256, beta=6.0, mc=256) ===
theta_max: min=0.346493  med=0.389858  max=0.479852
zmax:      min=0.0594305  med=0.0750371  max=0.112937
 r=0.25: cnt=0
 r=0.35: cnt=    2  min_ratio=99.4506  min_ratio_LCB=83.1626  p05=99.96  med=99.4506  worst_seed=4001450
 r=0.45: cnt=  248  min_ratio=82.841  min_ratio_LCB=66.2029  p05=91.4059  med=104.857  worst_seed=4001266
 r=0

In [5]:
import torch
import torch.nn as nn
import torch.optim as optim
import time
import math

# ==============================================================================
# ADVERSARIAL COERCIVITY SCANNER (STRICT PGD VERSION)
#
# Methodology:
# 1. Minimizes Ratio = P_Lambda(U) / Z2(U) to find counterexamples.
# 2. Uses Projected Gradient Descent (PGD) to strictly enforce the
#    small-field constraint (theta_max <= r).
# 3. Uses a Frozen Estimator to make the objective deterministic and differentiable.
# ==============================================================================

CONFIG = {
    'L': 8,                 # Lattice size
    'beta': 6.0,            # Inverse coupling
    'batch_size': 32,       # MEMORY SAFE: 32 parallel chains
    'n_steps': 200,         # Optimization steps
    'lr': 0.01,             # Learning rate
    'r_constraint': 0.6,    # Basin boundary (radians). Strict Limit.
    'mc_frozen': 64,        # MEMORY SAFE: 64 frozen directions
    'eps_fd': 1e-4,         # Finite difference epsilon
}

# --- SU(2) DIFFERENTIABLE OPS ---

def su2_mul(q1, q2):
    """Hamilton product of quaternions."""
    a1, b1, c1, d1 = q1.unbind(-1)
    a2, b2, c2, d2 = q2.unbind(-1)
    return torch.stack([
        a1*a2 - b1*b2 - c1*c2 - d1*d2,
        a1*b2 + b1*a2 + c1*d2 - d1*c2,
        a1*c2 - b1*d2 + c1*a2 + d1*b2,
        a1*d2 + b1*c2 - c1*b2 + d1*a2
    ], dim=-1)

def su2_conj(q):
    """Quaternion conjugate."""
    a, b, c, d = q.unbind(-1)
    return torch.stack([a, -b, -c, -d], dim=-1)

def su2_exp(v):
    """Exact exponential map from algebra (..., 3) to group (..., 4)."""
    theta = torch.linalg.norm(v, dim=-1, keepdim=True)

    # Differentiable sinc for numerical stability
    theta_sq = theta * theta
    sinc = torch.where(
        theta > 1e-6,
        torch.sin(theta) / theta,
        1.0 - theta_sq/6.0 + (theta_sq*theta_sq)/120.0
    )

    cos_t = torch.cos(theta)
    return torch.cat([cos_t, v * sinc], dim=-1)

def get_plaquette(U, mu, nu):
    """Compute plaquette U_{mu,nu} at all sites."""
    U_mu = U[..., mu, :]
    U_nu = U[..., nu, :]

    # Shift logic matches input dimensions
    # Dims: (..., X, Y, Z, T, mu, quat)
    U_nu_shift = torch.roll(U_nu, shifts=-1, dims=1+mu)
    U_mu_shift = torch.roll(U_mu, shifts=-1, dims=1+nu)

    term1 = su2_mul(U_mu, U_nu_shift)
    term2 = su2_mul(su2_conj(U_mu_shift), su2_conj(U_nu))
    return su2_mul(term1, term2)

def compute_trace_defects(U):
    """Returns z_p = 1 - Re(P_quat). Shape: (Batch, 6, L...)"""
    defects = []
    # 6 planes: (0,1), (0,2), (0,3), (1,2), (1,3), (2,3)
    for mu in range(4):
        for nu in range(mu+1, 4):
            P = get_plaquette(U, mu, nu)
            z = 1.0 - P[..., 0]
            defects.append(z)
    return torch.stack(defects, dim=1)

# --- FROZEN ESTIMATOR ---

class FrozenCoercivityEstimator:
    """
    Computes P_Lambda(U) using a fixed set of MC directions Xi.
    """
    def __init__(self, cfg, device):
        self.cfg = cfg
        self.device = device

        # Pre-generate frozen noise directions
        # Shape: (Mc, 1, L, L, L, L, 4, 3)
        shape = (cfg['mc_frozen'], 1, cfg['L'], cfg['L'], cfg['L'], cfg['L'], 4, 3)
        self.Xi = torch.randn(shape, device=device, dtype=torch.float32)

        # Pre-compute exp(± eps * Xi)
        eps = cfg['eps_fd']
        self.exp_p = su2_exp((eps * self.Xi).double())
        self.exp_m = su2_exp((-eps * self.Xi).double())

    def __call__(self, U):
        # U shape: (Batch, L, L, L, L, 4, 4)

        # 1. Base defects and Z2
        z_base = compute_trace_defects(U)
        z2 = torch.sum(z_base**2, dim=[1,2,3,4,5])
        z_max_val = torch.amax(z_base, dim=[1,2,3,4,5])

        # 2. P_Lambda Estimator
        # Broadcast U over MC dim: (1, Batch, ...)
        U_expanded = U.unsqueeze(0)

        # Perturb U -> (Mc, Batch, ...)
        U_p = su2_mul(U_expanded, self.exp_p)
        U_m = su2_mul(U_expanded, self.exp_m)

        # --- FLATTEN MC AND BATCH DIMS ---
        mc = U_p.shape[0]
        batch = U_p.shape[1]

        U_p_flat = U_p.reshape(mc * batch, *U_p.shape[2:])
        U_m_flat = U_m.reshape(mc * batch, *U_m.shape[2:])

        # Compute defects on flattened tensors
        z_p_flat = compute_trace_defects(U_p_flat)
        z_m_flat = compute_trace_defects(U_m_flat)

        # Reshape back to (Mc, Batch, ...)
        z_p = z_p_flat.view(mc, batch, *z_p_flat.shape[1:])
        z_m = z_m_flat.view(mc, batch, *z_m_flat.shape[1:])
        # --- END FIX ---

        # Finite Differences
        eps = self.cfg['eps_fd']
        Dz = (z_p - z_m) / (2 * eps)

        # D_Xi S = beta * sum_p (D_Xi z)
        DS = self.cfg['beta'] * torch.sum(Dz, dim=[2,3,4,5,6])

        # Inner product term: sum_p z_base * D_Xi z
        inner = torch.sum(z_base.unsqueeze(0) * Dz, dim=[2,3,4,5,6])

        # Sample estimate
        samples = DS * inner
        P_hat = torch.mean(samples, dim=0)

        return P_hat, z2, z_max_val

# --- OPTIMIZATION LOOP WITH PGD ---

def adversarial_scan():
    if not torch.cuda.is_available():
        print(">>> ERROR: CUDA NOT DETECTED.")
        return

    device = torch.device('cuda')
    print(f">>> ADVERSARIAL SCANNER [Device: {torch.cuda.get_device_name(0)}]")
    print(f">>> CONFIG: L={CONFIG['L']}, r={CONFIG['r_constraint']}, Batch={CONFIG['batch_size']}")
    print(f">>> MODE: Strict Projected Gradient Descent (PGD)")

    # 1. Initialize Estimator
    estimator = FrozenCoercivityEstimator(CONFIG, device)

    # 2. Initialize Algebra A (Batch, L, L, L, L, 4, 3)
    # Start safely inside basin (sigma=0.04)
    shape = (CONFIG['batch_size'], CONFIG['L'], CONFIG['L'], CONFIG['L'], CONFIG['L'], 4, 3)
    A = torch.randn(shape, device=device, dtype=torch.float64) * 0.04
    A.requires_grad_(True)

    # 3. Optimizer
    optimizer = optim.Adam([A], lr=CONFIG['lr'])

    worst_ratio = float('inf')
    t0 = time.time()

    print(f"\n{'STEP':<6} | {'MIN RATIO':<10} | {'MEAN RATIO':<10} | {'MAX THETA':<10} | {'WALL (s)':<8}")
    print("-" * 60)

    for step in range(CONFIG['n_steps']):
        optimizer.zero_grad()

        # Forward Map: A -> U
        U = su2_exp(A)

        # Evaluate Estimator
        P_hat, z2, z_max_batch = estimator(U)

        # Loss: Pure Minimization of Ratio
        ratio = P_hat / torch.clamp(z2, min=1e-12)
        loss = torch.sum(ratio)

        loss.backward()
        optimizer.step()

        # --- STRICT PROJECTION (PGD) ---
        # Force every link's algebra norm to be <= r_constraint
        with torch.no_grad():
            norms = torch.linalg.norm(A, dim=-1, keepdim=True)
            # Scale down if norm > r
            scale = torch.clamp(CONFIG['r_constraint'] / (norms + 1e-8), max=1.0)
            A.data *= scale

        # --- LOGGING ---
        with torch.no_grad():
            current_min = torch.min(ratio).item()
            current_mean = torch.mean(ratio).item()
            # Verify Theta Max (Should track r_constraint exactly if active)
            # Note: |A| maps directly to theta in su2_exp
            current_theta_max = torch.max(norms * scale).item()

            if current_min < worst_ratio:
                worst_ratio = current_min

            if step % 20 == 0:
                print(f"{step:<6} | {current_min:<10.4f} | {current_mean:<10.4f} | {current_theta_max:<10.4f} | {time.time()-t0:<8.1f}")

    # Final Report
    print("-" * 60)
    print(f">>> FINAL WORST RATIO FOUND: {worst_ratio:.4f}")

    if worst_ratio > 0:
        print(">>> RESULT: BASIN STABILITY CONFIRMED. (Adversary could not force ratio < 0 inside r)")
    else:
        print(">>> RESULT: COUNTEREXAMPLE FOUND. (Basin radius r is too large)")

if __name__ == "__main__":
    adversarial_scan()

>>> ADVERSARIAL SCANNER [Device: NVIDIA A100-SXM4-80GB]
>>> CONFIG: L=8, r=0.6, Batch=32
>>> MODE: Strict Projected Gradient Descent (PGD)

STEP   | MIN RATIO  | MEAN RATIO | MAX THETA  | WALL (s)
------------------------------------------------------------
0      | 79.2225    | 104.1848   | 0.2474     | 1.3     
20     | -3239.7405 | -2771.1568 | 0.5064     | 26.5    
40     | -5811.9169 | -5473.5943 | 0.6000     | 51.7    
60     | -7633.0872 | -7318.0585 | 0.6000     | 77.0    
80     | -9421.9037 | -9090.6154 | 0.6000     | 102.2   
100    | -10429.9254 | -9968.5543 | 0.6000     | 127.4   
120    | -11030.1508 | -10488.1186 | 0.6000     | 152.6   
140    | -11136.2493 | -10735.7251 | 0.6000     | 177.9   
160    | -11391.3718 | -10570.0854 | 0.6000     | 203.1   
180    | -11614.1023 | -10947.9478 | 0.6000     | 228.3   
------------------------------------------------------------
>>> FINAL WORST RATIO FOUND: -11639.8927
>>> RESULT: COUNTEREXAMPLE FOUND. (Basin radius r is too larg

In [8]:
import math
import torch
import torch.optim as optim

# --------------------------
# SU(2) utilities
# --------------------------
def su2_mul(q1, q2):
    a1,b1,c1,d1 = q1.unbind(-1)
    a2,b2,c2,d2 = q2.unbind(-1)
    return torch.stack([
        a1*a2 - b1*b2 - c1*c2 - d1*d2,
        a1*b2 + b1*a2 + c1*d2 - d1*c2,
        a1*c2 - b1*d2 + c1*a2 + d1*b2,
        a1*d2 + b1*c2 - c1*b2 + d1*a2,
    ], dim=-1)

def su2_conj(q):
    a,b,c,d = q.unbind(-1)
    return torch.stack([a, -b, -c, -d], dim=-1)

def su2_exp(v):
    theta = torch.linalg.norm(v, dim=-1, keepdim=True)
    theta2 = theta*theta
    s_over = torch.where(theta > 1e-8, torch.sin(theta)/theta, 1.0 - theta2/6.0 + theta2*theta2/120.0)
    return torch.cat([torch.cos(theta), s_over*v], dim=-1)

# --------------------------
# Plaquettes / defects
# U: (B,L,L,L,L,4,4)
# --------------------------
def plaquette(U, mu, nu):
    U_mu = U[..., mu, :]
    U_nu = U[..., nu, :]
    U_nu_xpmu = torch.roll(U_nu, shifts=-1, dims=1+mu)
    U_mu_xpnu = torch.roll(U_mu, shifts=-1, dims=1+nu)
    return su2_mul(
        su2_mul(
            su2_mul(U_mu, U_nu_xpmu),
            su2_conj(U_mu_xpnu)
        ),
        su2_conj(U_nu)
    )

def z_stack(U):
    zs = []
    for mu in range(4):
        for nu in range(mu+1, 4):
            P = plaquette(U, mu, nu)
            zs.append(1.0 - P[...,0])
    return torch.stack(zs, dim=1)  # (B,6,L,L,L,L)

def V_sum_z(U):
    z = z_stack(U)
    return z.sum(dim=(1,2,3,4,5))  # (B,)

# --------------------------
# Monte Carlo estimator for pairing:
# P = (1/2) <∇S, ∇V>
# with S = beta * sum z, V = sum z
# so S = beta * V  => ∇S = beta ∇V
# hence P = (beta/2) ||∇V||^2 >= 0 in the true model.
# We estimate ||∇V||^2 via E[(D V)^2].
# --------------------------
# REMOVED @torch.no_grad() to enable backpropagation
def estimate_pairing_ratio(U, beta, eps_fd, Xi_frozen, use_z2=True):
    # Xi_frozen: (mc,1,L,L,L,L,4,3) float32
    B = U.shape[0]
    mc = Xi_frozen.shape[0]
    dev = U.device

    V0 = V_sum_z(U)  # (B,)
    z0 = z_stack(U)
    z2 = (z0*z0).sum(dim=(1,2,3,4,5))  # (B,)

    Xi = Xi_frozen.expand(-1, B, -1, -1, -1, -1, -1, -1)  # (mc,B,L...,4,3)
    exp_p = su2_exp((eps_fd * Xi).to(U.dtype))
    exp_m = su2_exp((-eps_fd * Xi).to(U.dtype))

    U0 = U.unsqueeze(0)  # (1,B,...)
    Up = su2_mul(U0, exp_p)
    Um = su2_mul(U0, exp_m)

    # flatten mc*B for V eval
    Upf = Up.reshape(mc*B, *U.shape[1:])
    Umf = Um.reshape(mc*B, *U.shape[1:])

    Vp = V_sum_z(Upf).view(mc, B)
    Vm = V_sum_z(Umf).view(mc, B)

    dV = (Vp - Vm) / (2.0 * eps_fd)   # (mc,B)

    # ||∇V||^2 = E[(D V)^2] under isotropic tangent directions
    gradV2 = (dV*dV).mean(dim=0)      # (B,)

    # True pairing:
    # P = (1/2)<∇S,∇V> = (beta/2)||∇V||^2 >= 0
    P = 0.5 * beta * gradV2

    denom = z2 if use_z2 else torch.clamp(V0, 1e-12)
    ratio = P / torch.clamp(denom, 1e-12)
    return ratio

# --------------------------
# Adversarial radius sweep (PGD on A field)
# A: (B,L,L,L,L,4,3), U=exp(A)
# constraint: ||A|| <= r pointwise
# objective: minimize ratio (try to break coercivity in basin)
# --------------------------
def adversarial_radius_sweep(
    L=8, beta=6.0, radii=(0.1,0.2,0.3,0.4,0.6),
    batch=16, n_steps=120, lr=0.03, mc=64, eps_fd=1e-4, seed=0
):
    dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("[device]", dev)

    g = torch.Generator(device=dev).manual_seed(seed)

    Xi = torch.randn((mc,1,L,L,L,L,4,3), device=dev, dtype=torch.float32, generator=g)

    out = {}
    for r in radii:
        print(f"\n--- r = {r} ---")
        A = (0.5*r) * torch.randn((batch,L,L,L,L,4,3), device=dev, dtype=torch.float64, generator=g)
        A.requires_grad_(True)
        opt = optim.Adam([A], lr=lr)

        for _ in range(n_steps):
            opt.zero_grad()
            U = su2_exp(A)

            ratio = estimate_pairing_ratio(U, beta=beta, eps_fd=eps_fd, Xi_frozen=Xi, use_z2=True)
            # adversary wants MIN ratio -> minimize sum
            loss = ratio.sum()
            loss.backward()
            opt.step()

            # project back into pointwise ball ||A||<=r
            with torch.no_grad():
                nrm = torch.linalg.norm(A, dim=-1, keepdim=True)
                A.mul_(torch.clamp(r / (nrm + 1e-12), max=1.0))

        # final report
        with torch.no_grad():
            U = su2_exp(A)
            ratio = estimate_pairing_ratio(U, beta=beta, eps_fd=eps_fd, Xi_frozen=Xi, use_z2=True)
            mn = float(ratio.min().item())
            md = float(ratio.median().item())
        out[r] = (mn, md)
        print(f"min ratio = {mn:.6g}  | median ratio = {md:.6g}")

    print("\n=== SUMMARY (min, median) ===")
    for r,(mn,md) in out.items():
        print(f"r={r:<4}  min={mn: .6g}   med={md: .6g}")
    return out

# RUN
adversarial_radius_sweep()

[device] cuda

--- r = 0.1 ---
min ratio = 0.000213798  | median ratio = 0.000991239

--- r = 0.2 ---
min ratio = 6.24192e-06  | median ratio = 0.00177144

--- r = 0.3 ---
min ratio = 1.7795e-06  | median ratio = 0.000795841

--- r = 0.4 ---
min ratio = 2.92185e-05  | median ratio = 0.000165478

--- r = 0.6 ---
min ratio = 1.40698e-06  | median ratio = 2.12649e-05

=== SUMMARY (min, median) ===
r=0.1   min= 0.000213798   med= 0.000991239
r=0.2   min= 6.24192e-06   med= 0.00177144
r=0.3   min= 1.7795e-06   med= 0.000795841
r=0.4   min= 2.92185e-05   med= 0.000165478
r=0.6   min= 1.40698e-06   med= 2.12649e-05


{0.1: (0.00021379835813763222, 0.0009912387339827639),
 0.2: (6.24191920852102e-06, 0.0017714381860816104),
 0.3: (1.7794988212435488e-06, 0.0007958407651987318),
 0.4: (2.9218452146175576e-05, 0.00016547849939481314),
 0.6: (1.406975950911374e-06, 2.126489127985596e-05)}

In [9]:
import math
import time
import numpy as np
import torch
import torch.optim as optim

# ============================================================
# ADVERSARIAL RADIUS SWEEP — LINEAR COERCIVITY RATIO (FULL BLOCK)
# ------------------------------------------------------------
# Optimizes A (link algebra) with U = exp(A), projected into a
# plaquette-angle basin:  theta_max(U_p) <= r
#
# Uses the *Wilson-specialized true pairing surrogate*:
#   V(U) := sum_p z_p(U),   z_p = 1 - a_p
#   S(U) := beta * V(U)     =>  ∇S = beta ∇V
#   P(U) := (1/2)<∇S,∇V> = (beta/2)||∇V||^2  >= 0
#
# Estimates ||∇V||^2 via frozen MC directions Xi:
#   ||∇V||^2 ≈ E[(D_X V)^2]
#
# Scans radii r and adversarially minimizes:
#   ratio_lin(U) = P(U) / D(U)   where D(U)=sum_p z_p(U)
#
# This ratio matches the *linear* coercivity objective you actually use
# in the basin stability proof stack.
# ============================================================

CFG = dict(
    device="auto",     # "auto" | "cuda" | "cpu"
    L=8,
    beta=6.0,
    radii=[0.1, 0.2, 0.3, 0.4, 0.6],

    Ncand=16,
    steps=160,
    lr=3e-2,

    eps_fd=1e-4,
    mc_bank=64,
    mc_chunk=8,

    sigma_init=0.06,     # init amplitude for A
    clip_A_link=0.70,    # clamp ||A_link|| <= clip_A_link each projection
    proj_iters=8,
    proj_safety=0.985,
    theta_slack=1e-6,

    # denominator floors (prevents trivial vacuum wins)
    D_min=1e-6,
    Z2_min=1e-14,

    seed=1234,
)

# --------------------------
# SU(2) quaternion ops
# --------------------------
def su2_mul(q1, q2):
    a1,b1,c1,d1 = q1.unbind(-1)
    a2,b2,c2,d2 = q2.unbind(-1)
    return torch.stack([
        a1*a2 - b1*b2 - c1*c2 - d1*d2,
        a1*b2 + b1*a2 + c1*d2 - d1*c2,
        a1*c2 - b1*d2 + c1*a2 + d1*b2,
        a1*d2 + b1*c2 - c1*b2 + d1*a2,
    ], dim=-1)

def su2_conj(q):
    a,b,c,d = q.unbind(-1)
    return torch.stack([a, -b, -c, -d], dim=-1)

def su2_exp(v):
    theta = torch.linalg.norm(v, dim=-1, keepdim=True)
    theta2 = theta * theta
    s_over = torch.where(
        theta > 1e-8,
        torch.sin(theta) / theta.clamp_min(1e-30),
        1.0 - theta2/6.0 + (theta2*theta2)/120.0
    )
    return torch.cat([torch.cos(theta), s_over*v], dim=-1)

# --------------------------
# Plaquettes / defects
# U: (B,L,L,L,L,4,4)
# --------------------------
def plaquette(U, mu, nu):
    U_mu = U[..., mu, :]
    U_nu = U[..., nu, :]
    U_nu_xpmu = torch.roll(U_nu, shifts=-1, dims=1+mu)
    U_mu_xpnu = torch.roll(U_mu, shifts=-1, dims=1+nu)
    return su2_mul(
        su2_mul(
            su2_mul(U_mu, U_nu_xpmu),
            su2_conj(U_mu_xpnu)
        ),
        su2_conj(U_nu)
    )

def z_stack(U):
    zs = []
    for mu in range(4):
        for nu in range(mu+1, 4):
            P = plaquette(U, mu, nu)
            zs.append(1.0 - P[...,0])   # z_p = 1 - a_p
    return torch.stack(zs, dim=1)       # (B,6,L,L,L,L)

def V_sum_z(U):
    z = z_stack(U)
    return z.sum(dim=(1,2,3,4,5))       # (B,)

def Z2_sum_z2(U):
    z = z_stack(U)
    return (z*z).sum(dim=(1,2,3,4,5))   # (B,)

def theta_max_plaq(U):
    z = z_stack(U)
    ap = 1.0 - z
    ap = torch.clamp(ap, -1.0, 1.0)
    th = torch.acos(ap)  # (B,6,....)
    B = th.shape[0]
    return th.reshape(B, 6, -1).amax(dim=2).amax(dim=1)  # (B,)

# --------------------------
# Frozen MC bank: Xi (mc,1,L,L,L,L,4,3)
# --------------------------
def make_Xi_bank(mc, L, dev, seed=0):
    g = torch.Generator(device=dev).manual_seed(int(seed))
    return torch.randn((mc, 1, L, L, L, L, 4, 3), device=dev, dtype=torch.float32, generator=g)

# --------------------------
# Differentiable objective:
#   P = (beta/2) * E[(D_X V)^2]
#   ratio_lin = P / D
# --------------------------
def ratio_lin_from_frozen_Xi(U, beta, eps_fd, Xi_frozen, D_min):
    B = U.shape[0]
    mc = Xi_frozen.shape[0]

    D = V_sum_z(U)               # (B,)
    Xi = Xi_frozen.expand(-1, B, -1, -1, -1, -1, -1, -1)  # (mc,B,...,3)
    exp_p = su2_exp((eps_fd * Xi).to(U.dtype))
    exp_m = su2_exp((-eps_fd * Xi).to(U.dtype))

    U0 = U.unsqueeze(0)
    Up = su2_mul(U0, exp_p)
    Um = su2_mul(U0, exp_m)

    Upf = Up.reshape(mc*B, *U.shape[1:])
    Umf = Um.reshape(mc*B, *U.shape[1:])

    Vp = V_sum_z(Upf).view(mc, B)
    Vm = V_sum_z(Umf).view(mc, B)

    dV = (Vp - Vm) / (2.0 * eps_fd)
    gradV2 = (dV*dV).mean(dim=0)  # (B,)
    P = 0.5 * beta * gradV2       # (B,)

    ratio = P / torch.clamp(D, min=D_min)
    return ratio, D, P

# --------------------------
# Projection: enforce plaquette-angle basin by scaling A (per candidate)
# and clamp link algebra norm for stability.
# --------------------------
@torch.no_grad()
def project_A(A, r, clip_A_link, iters=8, safety=0.985):
    if clip_A_link is not None:
        n = torch.linalg.norm(A, dim=-1, keepdim=True).clamp_min(1e-12)
        A.mul_(torch.clamp(clip_A_link / n, max=1.0))

    for _ in range(iters):
        U = su2_exp(A).to(torch.float64)
        th = theta_max_plaq(U)
        viol = th > r
        if not bool(viol.any()):
            break
        s = (r / th.clamp_min(1e-12)) * safety
        s = torch.clamp(s, max=1.0)
        A[viol] *= s[viol].view(-1,1,1,1,1,1,1).to(A.dtype)

# --------------------------
# Sweep
# --------------------------
def adversarial_radius_sweep_linear():
    dev = torch.device("cuda" if torch.cuda.is_available() else "cpu") if CFG["device"]=="auto" else torch.device(CFG["device"])
    print("[device]", dev, (torch.cuda.get_device_name(0) if dev.type=="cuda" else ""))

    torch.manual_seed(int(CFG["seed"]))
    np.random.seed(int(CFG["seed"]))

    L = int(CFG["L"])
    beta = float(CFG["beta"])
    Ncand = int(CFG["Ncand"])
    eps_fd = float(CFG["eps_fd"])
    mc_bank = int(CFG["mc_bank"])
    mc_chunk = int(CFG["mc_chunk"])

    Xi = make_Xi_bank(mc_bank, L, dev, seed=int(CFG["seed"])+999)

    results = {}
    for r in CFG["radii"]:
        r = float(r)
        print(f"\n--- r = {r} ---")

        g = torch.Generator(device=dev).manual_seed(int(CFG["seed"]) + int(1e6*r))

        A = (CFG["sigma_init"] * torch.randn((Ncand, L,L,L,L, 4, 3), device=dev, dtype=torch.float64, generator=g)).requires_grad_(True)
        opt = optim.Adam([A], lr=float(CFG["lr"]), betas=(0.9, 0.999))

        with torch.no_grad():
            project_A(A, r=r, clip_A_link=CFG["clip_A_link"], iters=int(CFG["proj_iters"]), safety=float(CFG["proj_safety"]))

        best = dict(min=float("inf"), med=float("inf"), idx=-1, theta=float("nan"), D=float("nan"), Z2=float("nan"), P=float("nan"))

        t0 = time.time()
        for it in range(int(CFG["steps"])):
            opt.zero_grad(set_to_none=True)
            U = su2_exp(A)

            ratio, D, P = ratio_lin_from_frozen_Xi(
                U, beta=beta, eps_fd=eps_fd, Xi_frozen=Xi,
                D_min=float(CFG["D_min"])
            )

            loss = ratio.sum()
            loss.backward()
            opt.step()

            with torch.no_grad():
                project_A(A, r=r, clip_A_link=CFG["clip_A_link"], iters=int(CFG["proj_iters"]), safety=float(CFG["proj_safety"]))

                U2 = su2_exp(A).to(torch.float64)
                th2 = theta_max_plaq(U2)
                D2 = V_sum_z(U2)
                Z22 = Z2_sum_z2(U2)

                ratio2, D2b, P2b = ratio_lin_from_frozen_Xi(
                    U2, beta=beta, eps_fd=eps_fd, Xi_frozen=Xi,
                    D_min=float(CFG["D_min"])
                )

                feasible = (th2 <= r + float(CFG["theta_slack"])) & (D2 > float(CFG["D_min"])) & (Z22 > float(CFG["Z2_min"]))
                rr = ratio2.clone()
                rr[~feasible] = float("inf")

                mn = float(rr.min().item())
                md = float(rr.median().item())
                if mn < best["min"]:
                    j = int(torch.argmin(rr).item())
                    best = dict(
                        min=mn,
                        med=md,
                        idx=j,
                        theta=float(th2[j].item()),
                        D=float(D2[j].item()),
                        Z2=float(Z22[j].item()),
                        P=float(P2b[j].item()),
                    )

            if it % 20 == 0 or it == CFG["steps"]-1:
                with torch.no_grad():
                    thm = float(th2.min().item())
                    thd = float(th2.median().item())
                    thx = float(th2.max().item())
                print(f"[it {it:4d}] min={mn:.6g}  med={md:.6g}  theta(min/med/max)={thm:.4f}/{thd:.4f}/{thx:.4f}")

        wall = time.time() - t0
        results[r] = dict(best_min=best["min"], best_med=best["med"], best_idx=best["idx"],
                          theta=best["theta"], D=best["D"], Z2=best["Z2"], P=best["P"],
                          wall_s=wall)

        print(f"[r={r}] best_min={best['min']:.6g}  best_med={best['med']:.6g}  "
              f"(idx={best['idx']}, theta={best['theta']:.6g}, D={best['D']:.6g}, Z2={best['Z2']:.6g})  "
              f"wall={wall/60:.2f} min")

    print("\n=== SUMMARY ===")
    for r in CFG["radii"]:
        rr = float(r)
        v = results[rr]
        print(f"r={rr:<4}  best_min={v['best_min']: .6g}  best_med={v['best_med']: .6g}  "
              f"theta={v['theta']:.6g}  D={v['D']:.6g}  Z2={v['Z2']:.6g}  wall={v['wall_s']/60:.2f} min")

    return results

# RUN
results = adversarial_radius_sweep_linear()


[device] cuda NVIDIA A100-SXM4-80GB

--- r = 0.1 ---
[it    0] min=16567.1  med=17090.9  theta(min/med/max)=0.0984/0.0985/0.0985
[it   20] min=0.0010738  med=0.107441  theta(min/med/max)=0.0985/0.0985/0.0985
[it   40] min=1.37651  med=297.352  theta(min/med/max)=0.0984/0.0985/0.0988
[it   60] min=0.0369869  med=273.106  theta(min/med/max)=0.0984/0.0985/0.0985
[it   80] min=1.48849  med=516.2  theta(min/med/max)=0.0984/0.0985/0.0986
[it  100] min=10.8699  med=75.8032  theta(min/med/max)=0.0984/0.0985/0.0986
[it  120] min=2.23542  med=53.4715  theta(min/med/max)=0.0985/0.0985/0.0986
[it  140] min=3.36952  med=33.9425  theta(min/med/max)=0.0985/0.0985/0.0986
[it  159] min=0.283258  med=13.2193  theta(min/med/max)=0.0985/0.0985/0.0986
[r=0.1] best_min=0.000367027  best_med=2.22797  (idx=0, theta=0.0984993, D=28.945, Z2=0.0490204)  wall=2.60 min

--- r = 0.2 ---
[it    0] min=12778  med=13246  theta(min/med/max)=0.1965/0.1969/0.1972
[it   20] min=0.0118555  med=0.999559  theta(min/med/max)=

In [11]:
import math
import time
import numpy as np
import torch
import torch.optim as optim

# ============================================================
# ADVERSARIAL RADIUS SWEEP — QUARTIC (APPENDIX G/I-ALIGNED)
# ------------------------------------------------------------
# Targets the quartic basin coercivity quantity consistent with:
#   V(U) = sum_p z_p(U)^2
#   S_W(U) = beta * sum_p z_p(U)
#   Pairing term:  P(U) = (1/2) <∇S_W, ∇V>
#
# Monte Carlo identity on configuration space:
#   <∇S,∇V> = E_X [ (D_X S)(D_X V) ]   for isotropic tangent directions X
#
# We minimize over U in basin: theta_max(U_p) <= r:
#   ratio_q(U) := P_hat(U) / Z2(U),   Z2(U) = sum_p z_p(U)^2 = V(U)
#
# Optimization: frozen MC bank (deterministic objective)
# Audit: fresh MC (large), mean ± SE, and 2σ lower confidence bound
# ============================================================

CFG = dict(
    device="auto",        # "auto" | "cuda" | "cpu"
    L=8,
    beta=6.0,
    radii=[0.10, 0.20, 0.30, 0.40, 0.60],

    Ncand=12,             # parallel candidates
    steps=180,
    lr=2e-2,

    # finite difference + MC
    eps_fd=1e-4,
    mc_bank=64,
    mc_chunk_bank=8,

    # audit MC (fresh directions)
    mc_audit=2048,
    mc_chunk_audit=32,

    # init and constraints
    sigma_init=0.03,      # init A ~ N(0, sigma_init^2)
    clip_A_link=0.60,     # safety clamp on ||A_link||
    proj_iters=8,
    proj_safety=0.985,
    theta_slack=1e-6,

    # denominator floor
    Z2_min=1e-14,

    seed=1234,
    out_npz="quartic_basin_adversary_RESULTS.npz",
)

# --------------------------
# SU(2) quaternion ops
# --------------------------
def su2_mul(q1, q2):
    a1,b1,c1,d1 = q1.unbind(-1)
    a2,b2,c2,d2 = q2.unbind(-1)
    return torch.stack([
        a1*a2 - b1*b2 - c1*c2 - d1*d2,
        a1*b2 + b1*a2 + c1*d2 - d1*c2,
        a1*c2 - b1*d2 + c1*a2 + d1*b2,
        a1*d2 + b1*c2 - c1*b2 + d1*a2,
    ], dim=-1)

def su2_conj(q):
    a,b,c,d = q.unbind(-1)
    return torch.stack([a, -b, -c, -d], dim=-1)

def su2_exp(v):
    theta = torch.linalg.norm(v, dim=-1, keepdim=True)
    theta2 = theta * theta
    s_over = torch.where(
        theta > 1e-8,
        torch.sin(theta) / theta.clamp_min(1e-30),
        1.0 - theta2/6.0 + (theta2*theta2)/120.0
    )
    return torch.cat([torch.cos(theta), s_over*v], dim=-1)

# --------------------------
# Lattice ops (4D torus)
# U: (B,L,L,L,L,4,4)
# --------------------------
def plaquette(U, mu, nu):
    U_mu = U[..., mu, :]
    U_nu = U[..., nu, :]
    U_nu_xpmu = torch.roll(U_nu, shifts=-1, dims=1+mu)
    U_mu_xpnu = torch.roll(U_mu, shifts=-1, dims=1+nu)
    return su2_mul(
        su2_mul(
            su2_mul(U_mu, U_nu_xpmu),
            su2_conj(U_mu_xpnu)
        ),
        su2_conj(U_nu)
    )

def z_stack(U):
    zs = []
    for mu in range(4):
        for nu in range(mu+1, 4):
            P = plaquette(U, mu, nu)
            zs.append(1.0 - P[...,0])   # z_p = 1 - a_p
    return torch.stack(zs, dim=1)       # (B,6,L,L,L,L)

def compute_stats(U):
    # returns: zflat (B,Np), D (B,), Z2 (B,), zmax (B,), theta_max (B,)
    z = z_stack(U)                                # (B,6,...)
    B = z.shape[0]
    zflat = z.reshape(B, 6, -1).reshape(B, -1)    # (B,Np)
    D = zflat.sum(dim=1)                          # (B,)
    Z2 = (zflat*zflat).sum(dim=1)                 # (B,)
    zmax = zflat.max(dim=1).values                # (B,)
    ap = 1.0 - z
    ap = torch.clamp(ap, -1.0, 1.0)
    theta = torch.acos(ap)                        # (B,6,...)
    theta_max = theta.reshape(B, 6, -1).amax(dim=2).amax(dim=1)
    return zflat, D, Z2, zmax, theta_max

# --------------------------
# Frozen MC bank: Xi (mc,1,L,L,L,L,4,3)
# --------------------------
def make_Xi_bank(mc, L, dev, seed=0):
    g = torch.Generator(device=dev).manual_seed(int(seed))
    return torch.randn((mc, 1, L, L, L, L, 4, 3), device=dev, dtype=torch.float32, generator=g)

# --------------------------
# Quartic pairing estimator (differentiable, frozen bank)
#
# V = sum z^2
# D_X V = sum_p 2 z_p D_X z_p
# S = beta sum z
# D_X S = beta sum_p D_X z_p
#
# <∇S,∇V> = E[ (D_X S)(D_X V) ]
# P = (1/2) <∇S,∇V>
# ratio_q = P / Z2
# --------------------------
def ratio_quartic_from_bank(U, beta, eps_fd, Xi_frozen, mc_chunk, Z2_min):
    device = U.device
    dtype = U.dtype
    mc = Xi_frozen.shape[0]
    B = U.shape[0]

    # base z, Z2
    z0 = z_stack(U)                                 # (B,6,...)
    z0flat = z0.reshape(B, 6, -1).reshape(B, -1)    # (B,Np)
    Z2 = (z0flat*z0flat).sum(dim=1)                 # (B,)

    Xi = Xi_frozen.expand(-1, B, -1, -1, -1, -1, -1, -1).to(dtype)  # (mc,B,...,3)

    U0 = U.unsqueeze(0)                              # (1,B,...)
    sumP = torch.zeros((B,), device=device, dtype=torch.float64)

    done = 0
    while done < mc:
        k = min(mc_chunk, mc - done)
        Xi_k = Xi[done:done+k]                      # (k,B,...,3)

        exp_p = su2_exp(eps_fd * Xi_k)
        exp_m = su2_exp(-eps_fd * Xi_k)

        Up = su2_mul(U0, exp_p)                     # (k,B,...)
        Um = su2_mul(U0, exp_m)

        # z at perturbed configs
        zp = z_stack(Up.reshape(k*B, *U.shape[1:])) # (k*B,6,...)
        zm = z_stack(Um.reshape(k*B, *U.shape[1:]))

        zp_flat = zp.reshape(k*B, 6, -1).reshape(k*B, -1).view(k, B, -1)  # (k,B,Np)
        zm_flat = zm.reshape(k*B, 6, -1).reshape(k*B, -1).view(k, B, -1)

        dz = (zp_flat - zm_flat) / (2.0 * eps_fd)                         # (k,B,Np)

        # D_X S = beta * sum_p D_X z_p
        dS = beta * (dz.sum(dim=2))                                       # (k,B)

        # D_X V = sum_p 2 z_p D_X z_p   (use base z0flat for the bilinear form)
        dV = (2.0 * (dz * z0flat.unsqueeze(0))).sum(dim=2)                # (k,B)

        # P = (1/2) E[dS*dV]
        Pk = 0.5 * (dS * dV)                                              # (k,B)
        sumP += Pk.sum(dim=0).to(torch.float64)

        done += k
        del Xi_k, exp_p, exp_m, Up, Um, zp, zm, zp_flat, zm_flat, dz, dS, dV, Pk
        if device.type == "cuda":
            torch.cuda.empty_cache()

    P_hat = sumP / float(mc)                                              # (B,)
    ratio = P_hat.to(torch.float64) / torch.clamp(Z2.to(torch.float64), min=Z2_min)
    return ratio, P_hat, Z2

# --------------------------
# Projection to plaquette-angle basin by scaling A (per candidate)
# with link clamp for safety.
# --------------------------
@torch.no_grad()
def project_A_to_theta_basin(A, r, clip_A_link, iters=8, safety=0.985):
    if clip_A_link is not None:
        n = torch.linalg.norm(A, dim=-1, keepdim=True).clamp_min(1e-12)
        A.mul_(torch.clamp(clip_A_link / n, max=1.0))

    for _ in range(iters):
        U = su2_exp(A).to(torch.float64)
        _, _, _, _, th = compute_stats(U)
        viol = th > r
        if not bool(viol.any()):
            break
        s = (r / th.clamp_min(1e-12)) * safety
        s = torch.clamp(s, max=1.0)
        A[viol] *= s[viol].view(-1,1,1,1,1,1,1).to(A.dtype)

# --------------------------
# High-MC fresh audit of the best candidate (mean ± SE, 2σ LCB)
# --------------------------
@torch.no_grad()
def audit_best(U, beta, eps_fd, mc, mc_chunk, seed, Z2_min):
    device = U.device
    B = U.shape[0]
    assert B == 1

    zflat, D, Z2, zmax, th = compute_stats(U)
    z0flat = zflat  # (1,Np)

    g = torch.Generator(device=device).manual_seed(int(seed))

    sumP = torch.zeros((1,), device=device, dtype=torch.float64)
    sumP2 = torch.zeros((1,), device=device, dtype=torch.float64)

    done = 0
    U0 = U.unsqueeze(0)  # (1,1,...)
    while done < mc:
        k = min(mc_chunk, mc - done)
        Xi = torch.randn((k, 1, *U.shape[1:5], 4, 3), device=device, dtype=torch.float32, generator=g).to(torch.float64)

        exp_p = su2_exp(eps_fd * Xi)
        exp_m = su2_exp(-eps_fd * Xi)

        Up = su2_mul(U0, exp_p)  # (k,1,...)
        Um = su2_mul(U0, exp_m)

        zp = z_stack(Up.reshape(k, *U.shape[1:]))  # (k,6,...)
        zm = z_stack(Um.reshape(k, *U.shape[1:]))

        zp_flat = zp.reshape(k, 6, -1).reshape(k, -1)  # (k,Np)
        zm_flat = zm.reshape(k, 6, -1).reshape(k, -1)

        dz = (zp_flat - zm_flat) / (2.0 * eps_fd)       # (k,Np)

        dS = beta * dz.sum(dim=1)                        # (k,)
        dV = (2.0 * (dz * z0flat.squeeze(0))).sum(dim=1) # (k,)
        Pk = 0.5 * (dS * dV)                             # (k,)

        sumP += Pk.sum().view(1).to(torch.float64)
        sumP2 += (Pk*Pk).sum().view(1).to(torch.float64)

        done += k
        del Xi, exp_p, exp_m, Up, Um, zp, zm, zp_flat, zm_flat, dz, dS, dV, Pk
        if device.type == "cuda":
            torch.cuda.empty_cache()

    P_hat = sumP / float(mc)
    var = (sumP2 - (sumP*sumP)/float(mc)) / float(max(1, mc-1))
    var = torch.clamp(var, min=0.0)
    P_se = torch.sqrt(var / float(mc))

    ratio = P_hat / torch.clamp(Z2.to(torch.float64), min=Z2_min)
    ratio_lcb = (P_hat - 2.0*P_se) / torch.clamp(Z2.to(torch.float64), min=Z2_min)

    return dict(
        D=float(D.item()),
        Z2=float(Z2.item()),
        zmax=float(zmax.item()),
        theta_max=float(th.item()),
        P_hat=float(P_hat.item()),
        P_se=float(P_se.item()),
        ratio=float(ratio.item()),
        ratio_lcb=float(ratio_lcb.item()),
    )

# --------------------------
# Main sweep
# --------------------------
def run_quartic_adversarial_sweep():
    dev = torch.device("cuda" if torch.cuda.is_available() else "cpu") if CFG["device"]=="auto" else torch.device(CFG["device"])
    print("[device]", dev, (torch.cuda.get_device_name(0) if dev.type=="cuda" else ""))

    torch.manual_seed(int(CFG["seed"]))
    np.random.seed(int(CFG["seed"]))

    L = int(CFG["L"])
    beta = float(CFG["beta"])
    eps_fd = float(CFG["eps_fd"])
    Ncand = int(CFG["Ncand"])

    Xi_bank = make_Xi_bank(int(CFG["mc_bank"]), L, dev, seed=int(CFG["seed"])+999)

    rows = []
    for r in CFG["radii"]:
        r = float(r)
        print(f"\n--- r = {r} ---")

        g = torch.Generator(device=dev).manual_seed(int(CFG["seed"]) + int(1e6*r))
        A = (CFG["sigma_init"] * torch.randn((Ncand, L,L,L,L, 4, 3), device=dev, dtype=torch.float64, generator=g)).requires_grad_(True)
        opt = optim.Adam([A], lr=float(CFG["lr"]), betas=(0.9, 0.999))

        with torch.no_grad():
            project_A_to_theta_basin(A, r=r, clip_A_link=float(CFG["clip_A_link"]), iters=int(CFG["proj_iters"]), safety=float(CFG["proj_safety"]))

        best = dict(val=float("inf"), idx=-1, A_best=None, D=None, Z2=None, zmax=None, th=None, P=None)

        t0 = time.time()
        for it in range(int(CFG["steps"])):
            opt.zero_grad(set_to_none=True)
            U = su2_exp(A)

            ratio, P_hat, Z2 = ratio_quartic_from_bank(
                U, beta=beta, eps_fd=eps_fd,
                Xi_frozen=Xi_bank, mc_chunk=int(CFG["mc_chunk_bank"]),
                Z2_min=float(CFG["Z2_min"])
            )
            loss = ratio.sum()
            loss.backward()
            opt.step()

            with torch.no_grad():
                project_A_to_theta_basin(A, r=r, clip_A_link=float(CFG["clip_A_link"]), iters=int(CFG["proj_iters"]), safety=float(CFG["proj_safety"]))

                U2 = su2_exp(A).to(torch.float64)
                zflat2, D2, Z22, zmax2, th2 = compute_stats(U2)
                ratio2, P2, Z2b = ratio_quartic_from_bank(
                    U2, beta=beta, eps_fd=eps_fd,
                    Xi_frozen=Xi_bank, mc_chunk=int(CFG["mc_chunk_bank"]),
                    Z2_min=float(CFG["Z2_min"])
                )

                feasible = (th2 <= r + float(CFG["theta_slack"])) & (Z22 > float(CFG["Z2_min"]))
                rr = ratio2.clone()
                rr[~feasible] = float("inf")

                mn = float(rr.min().item())
                md = float(rr.median().item())
                if mn < best["val"]:
                    j = int(torch.argmin(rr).item())
                    best["val"] = mn
                    best["idx"] = j
                    best["A_best"] = A[j].detach().clone()
                    best["D"] = float(D2[j].item())
                    best["Z2"] = float(Z22[j].item())
                    best["zmax"] = float(zmax2[j].item())
                    best["th"] = float(th2[j].item())
                    best["P"] = float(P2[j].item())

            if it % 20 == 0 or it == CFG["steps"]-1:
                thm = float(th2.min().item())
                thd = float(th2.median().item())
                thx = float(th2.max().item())
                print(f"[it {it:4d}] min={mn:.6g}  med={md:.6g}  theta(min/med/max)={thm:.4f}/{thd:.4f}/{thx:.4f}")

            del U, ratio, P_hat, Z2
            if dev.type == "cuda":
                torch.cuda.empty_cache()

        wall = time.time() - t0
        print(f"[r={r}] best_min(bank)={best['val']:.6g}  idx={best['idx']}  theta={best['th']:.6g}  D={best['D']:.6g}  Z2={best['Z2']:.6g}  wall={wall/60:.2f} min")

        # High-MC fresh audit (single candidate)
        A_best = best["A_best"].unsqueeze(0).to(torch.float64)
        U_best = su2_exp(A_best)  # (1,L,L,L,L,4,4)
        audit = audit_best(
            U_best, beta=beta, eps_fd=eps_fd,
            mc=int(CFG["mc_audit"]), mc_chunk=int(CFG["mc_chunk_audit"]),
            seed=int(CFG["seed"]) + 424242 + int(1e6*r),
            Z2_min=float(CFG["Z2_min"]),
        )

        print("  [audit] ratio_mean   =", f"{audit['ratio']:.6g}")
        print("  [audit] ratio_LCB(2σ)=", f"{audit['ratio_lcb']:.6g}")
        print("  [audit] P_hat ± SE    =", f"{audit['P_hat']:.6e} ± {audit['P_se']:.2e}")
        print("  [audit] D, Z2, zmax   =", f"{audit['D']:.6g}", f"{audit['Z2']:.6g}", f"{audit['zmax']:.6g}")
        print("  [audit] theta_max     =", f"{audit['theta_max']:.6g}")

        rows.append(dict(
            r=r, beta=beta, L=L,
            best_bank_ratio=best["val"],
            best_theta=best["th"], best_D=best["D"], best_Z2=best["Z2"], best_zmax=best["zmax"],
            audit_ratio=audit["ratio"], audit_ratio_lcb=audit["ratio_lcb"],
            audit_P_hat=audit["P_hat"], audit_P_se=audit["P_se"],
            audit_D=audit["D"], audit_Z2=audit["Z2"], audit_zmax=audit["zmax"], audit_theta=audit["theta_max"],
            wall_s=wall,
        ))

    # Save
    out = CFG.get("out_npz", None)
    if out:
        np.savez_compressed(out, CFG_text=str(CFG), rows=np.array(rows, dtype=object))
        print(f"\n[saved] {out}")

    print("\n=== FINAL TABLE (audit values) ===")
    for row in rows:
        print(f"r={row['r']:<4}  ratio={row['audit_ratio']:.6g}  LCB={row['audit_ratio_lcb']:.6g}  "
              f"D={row['audit_D']:.3g}  Z2={row['audit_Z2']:.3g}  theta={row['audit_theta']:.4f}")

run_quartic_adversarial_sweep()


[device] cuda NVIDIA A100-SXM4-80GB

--- r = 0.1 ---
[it    0] min=25734  med=26350.3  theta(min/med/max)=0.0985/0.0985/0.0987
[it   20] min=-7645.53  med=-6562.61  theta(min/med/max)=0.0985/0.0985/0.0985
[it   40] min=-7760.67  med=-6808.65  theta(min/med/max)=0.0985/0.0985/0.0985
[it   60] min=-8467.33  med=-7203.13  theta(min/med/max)=0.0985/0.0985/0.0985
[it   80] min=-8830.15  med=-7187.68  theta(min/med/max)=0.0985/0.0985/0.0985
[it  100] min=-8599.58  med=-6476.36  theta(min/med/max)=0.0985/0.0985/0.0985
[it  120] min=-9022.93  med=-7072.15  theta(min/med/max)=0.0985/0.0985/0.0985
[it  140] min=-8968.97  med=-6544.17  theta(min/med/max)=0.0985/0.0985/0.0985
[it  160] min=-8629.83  med=-6647.93  theta(min/med/max)=0.0985/0.0985/0.0985
[it  179] min=-8902.39  med=-6791.02  theta(min/med/max)=0.0985/0.0985/0.0985
[r=0.1] best_min(bank)=-9287.47  idx=7  theta=0.0985003  D=0.605478  Z2=0.000106239  wall=2.81 min
  [audit] ratio_mean   = 102.317
  [audit] ratio_LCB(2σ)= 89.495
  [audi

KeyboardInterrupt: 

In [13]:
import math
import time
import numpy as np
import torch

# ============================================================
# QUARTIC BASIN COERCIVITY — ADVERSARIAL RADIUS SWEEP (v4 FINAL)
#
# For each radius r in r_list, adversarially minimize on K(r):
#   ratio(U) = P_hat(U) / Z2(U)
# where
#   z_p(U)   = 1 - a_p(U)   (a_p is scalar part of plaquette quaternion)
#   Z2(U)    = sum_p z_p(U)^2
#   S(U)     = beta * sum_p z_p(U)   (Wilson)
#   P_hat(U) = E_X [ (D_X S) * sum_p z_p(U) (D_X z_p(U)) ]
#
# TRAIN uses seeded chunk bank seed0+100000, VAL uses seed0+200000.
# High-MC audit uses fresh RNG for mean±SE and 2σ LCB.
# ============================================================

CFG = dict(
    device="auto",          # "auto" | "cuda" | "cpu"
    L=8,                    # even; try 12/16 on A100
    beta=6.0,

    r_list=[0.10, 0.20, 0.25, 0.30, 0.40, 0.60],

    Ncand=16,
    steps=250,
    lr=6e-3,
    grad_clip=1.0,
    adam_beta1=0.9,
    adam_beta2=0.999,

    eps_fd=1e-4,

    # TRAIN/VAL (deterministic, seeded chunks)
    mc_train=64,
    mc_val=64,
    mc_chunk=8,
    val_every=10,

    # Final audit (fresh MC)
    mc_eval=1024,
    mc_chunk_eval=16,
    nsigma_eval=2.0,

    sigma_init=0.02,

    # Basin projection knobs
    clip_A_link=0.35,       # clamp |A_link| <= this (radians)
    proj_iters=8,
    proj_safety=0.985,
    theta_slack=1e-6,

    # denom/robustness
    z2_floor_opt=1e-10,     # denom floor during optimization
    z2_min_eval=1e-14,      # denom floor during reporting
    z2_penalty=1e-4,        # discourages z2->0 games

    # candidate recovery
    reinit_sigma=0.02,
    reinit_absmax=50.0,     # if |A| max exceeds this, reinit candidate

    seed=1234,
    out_npz="quartic_basin_radius_sweep_v4_final.npz",
)

# --------------------------
# SU(2) quaternion ops
# --------------------------
def su2_mul(q1, q2):
    a1,b1,c1,d1 = q1.unbind(-1)
    a2,b2,c2,d2 = q2.unbind(-1)
    return torch.stack([
        a1*a2 - b1*b2 - c1*c2 - d1*d2,
        a1*b2 + b1*a2 + c1*d2 - d1*c2,
        a1*c2 - b1*d2 + c1*a2 + d1*b2,
        a1*d2 + b1*c2 - c1*b2 + d1*a2,
    ], dim=-1)

def su2_conj(q):
    a,b,c,d = q.unbind(-1)
    return torch.stack([a, -b, -c, -d], dim=-1)

def su2_exp(v):
    theta = torch.linalg.norm(v, dim=-1, keepdim=True)
    theta2 = theta*theta
    s_over = torch.where(
        theta > 1e-8,
        torch.sin(theta)/theta.clamp_min(1e-30),
        1.0 - theta2/6.0 + (theta2*theta2)/120.0
    )
    return torch.cat([torch.cos(theta), s_over*v], dim=-1)

# --------------------------
# Lattice ops (4D torus)
# U: (B,L,L,L,L,4,4)
# --------------------------
def roll4(x, mu, shift):
    start = x.ndim - 1 - 4
    return torch.roll(x, shifts=shift, dims=start + mu)

def plaquette(U, mu, nu):
    U_mu = U[..., mu, :]
    U_nu = U[..., nu, :]
    U_nu_xpmu = roll4(U_nu, mu, -1)
    U_mu_xpnu = roll4(U_mu, nu, -1)
    return su2_mul(
        su2_mul(su2_mul(U_mu, U_nu_xpmu), su2_conj(U_mu_xpnu)),
        su2_conj(U_nu)
    )

def all_defects(U):
    # z_p = 1 - a_p ; clamp a_p for numerical safety
    zs = []
    for mu in range(4):
        for nu in range(mu+1, 4):
            P = plaquette(U, mu, nu)
            ap = torch.clamp(P[..., 0], -1.0, 1.0)
            zs.append(1.0 - ap)
    return torch.stack(zs, dim=0)  # (6,B,...) or (6,k,B,...)

def compute_zflat_z2_zmax(U):
    z = all_defects(U)  # (6,B,L,L,L,L)
    B = U.shape[0]
    zflat = z.reshape(6, B, -1).permute(1, 0, 2).reshape(B, -1)  # (B,Np)
    z2 = (zflat * zflat).sum(dim=1)
    zmax = zflat.max(dim=1).values
    # sanitize
    z2 = torch.where(torch.isfinite(z2), z2, torch.zeros_like(z2))
    zmax = torch.where(torch.isfinite(zmax), zmax, torch.zeros_like(zmax))
    zflat = torch.where(torch.isfinite(zflat), zflat, torch.zeros_like(zflat))
    return zflat, z2, zmax

def theta_max_plaq(U):
    z = all_defects(U)  # (6,B,...)
    B = U.shape[0]
    ap = torch.clamp(1.0 - z, -1.0, 1.0)
    th = torch.acos(ap)
    theta_max = th.reshape(6, B, -1).amax(dim=2).amax(dim=0)
    theta_max = torch.where(torch.isfinite(theta_max), theta_max, torch.full_like(theta_max, float("inf")))
    return theta_max

# --------------------------
# Candidate recovery
# --------------------------
@torch.no_grad()
def sanitize_and_reinit(A, gen, sigma, absmax):
    # A: (Ncand,L,L,L,L,4,3)
    bad = ~torch.isfinite(A).all(dim=(1,2,3,4,5,6))
    if absmax is not None:
        too_big = A.abs().amax(dim=(1,2,3,4,5,6)) > float(absmax)
        bad = bad | too_big
    if bool(bad.any()):
        A[bad] = sigma * torch.randn_like(A[bad], generator=gen)

# --------------------------
# Projection to basin: theta_max(U_p) <= r
# --------------------------
@torch.no_grad()
def project_to_basin(A, r, clip_A_link, iters, safety):
    if clip_A_link is not None:
        n = torch.linalg.norm(A, dim=-1, keepdim=True).clamp_min(1e-12)
        A.mul_(torch.clamp(float(clip_A_link) / n, max=1.0))

    for _ in range(int(iters)):
        U = su2_exp(A)
        theta_max = theta_max_plaq(U)
        # if theta_max is inf for some candidate, crush it to zero
        bad = torch.isinf(theta_max)
        if bool(bad.any()):
            A[bad] *= 0.0
            theta_max = theta_max_plaq(su2_exp(A))

        viol = theta_max > float(r)
        if not bool(viol.any()):
            break
        s = (float(r) / theta_max.clamp_min(1e-12)) * float(safety)
        s = torch.clamp(s, max=1.0)
        A[viol] *= s[viol].view(-1,1,1,1,1,1,1).to(A.dtype)

# --------------------------
# TRAIN/VAL: deterministic seeded bank via per-chunk seeding
# --------------------------
def P_hat_from_seeded_bank(U, z0flat, beta, eps_fd, mc, mc_chunk, seed_bank):
    device = U.device
    dtype = U.dtype
    B = U.shape[0]
    U0 = U.unsqueeze(0)

    sumP = torch.zeros((B,), device=device, dtype=torch.float64)

    done = 0
    mc = int(mc)
    mc_chunk = int(mc_chunk)
    while done < mc:
        k = min(mc_chunk, mc - done)
        g = torch.Generator(device=device)
        g.manual_seed(int(seed_bank + done))
        Xi = torch.randn((k, *U.shape[:5], 4, 3), device=device, dtype=torch.float32, generator=g).to(dtype)

        exp_p = su2_exp(float(eps_fd) * Xi)
        exp_m = su2_exp(-float(eps_fd) * Xi)

        Up = su2_mul(U0, exp_p)
        Um = su2_mul(U0, exp_m)

        zp = all_defects(Up)  # (6,k,B,...)
        zm = all_defects(Um)

        zp_flat = zp.reshape(6, k, B, -1).permute(1, 2, 0, 3).reshape(k, B, -1)
        zm_flat = zm.reshape(6, k, B, -1).permute(1, 2, 0, 3).reshape(k, B, -1)

        dz = (zp_flat - zm_flat) / (2.0 * float(eps_fd))
        dS = float(beta) * (zp_flat.sum(dim=2) - zm_flat.sum(dim=2)) / (2.0 * float(eps_fd))

        inner = (dz * z0flat.unsqueeze(0)).sum(dim=2)
        Pk = dS * inner

        sumP += Pk.sum(dim=0).to(torch.float64)

        done += k

    return sumP / float(mc)

# --------------------------
# High-MC audit with SE and LCB
# --------------------------
@torch.no_grad()
def evaluate_high_mc(U, beta, eps_fd, mc, mc_chunk, seed, z2_min, nsigma):
    device = U.device
    zflat, z2, zmax = compute_zflat_z2_zmax(U)
    theta_max = theta_max_plaq(U)

    g = torch.Generator(device=device)
    g.manual_seed(int(seed))

    sumP = torch.zeros((1,), device=device, dtype=torch.float64)
    sumP2 = torch.zeros((1,), device=device, dtype=torch.float64)

    done = 0
    U0 = U.unsqueeze(0)
    mc = int(mc)
    mc_chunk = int(mc_chunk)
    while done < mc:
        k = min(mc_chunk, mc - done)
        Xi = torch.randn((k, *U.shape[:5], 4, 3), device=device, dtype=torch.float32, generator=g).to(U.dtype)

        exp_p = su2_exp(float(eps_fd) * Xi)
        exp_m = su2_exp(-float(eps_fd) * Xi)

        Up = su2_mul(U0, exp_p)
        Um = su2_mul(U0, exp_m)

        zp = all_defects(Up)
        zm = all_defects(Um)

        zp_flat = zp.reshape(6, k, 1, -1).permute(1, 2, 0, 3).reshape(k, 1, -1)
        zm_flat = zm.reshape(6, k, 1, -1).permute(1, 2, 0, 3).reshape(k, 1, -1)

        dz = (zp_flat - zm_flat) / (2.0 * float(eps_fd))
        dS = float(beta) * (zp_flat.sum(dim=2) - zm_flat.sum(dim=2)) / (2.0 * float(eps_fd))

        inner = (dz * zflat.unsqueeze(0)).sum(dim=2)
        Pk = (dS * inner).squeeze(1)  # (k,)

        sumP += Pk.sum().to(torch.float64)
        sumP2 += (Pk * Pk).sum().to(torch.float64)

        done += k

    P_hat = (sumP / float(mc)).item()
    var = (sumP2.item() - (sumP.item() * sumP.item()) / float(mc)) / float(max(1, mc - 1))
    var = max(0.0, var)
    P_se = math.sqrt(var / float(mc))

    Z2 = float(z2[0].item())
    denom = max(float(z2_min), Z2)
    ratio = P_hat / denom
    ratio_lcb = (P_hat - float(nsigma) * P_se) / denom

    return dict(
        P_hat=float(P_hat),
        P_se=float(P_se),
        Z2=float(Z2),
        zmax=float(zmax[0].item()),
        theta_max=float(theta_max[0].item()),
        ratio=float(ratio),
        ratio_lcb=float(ratio_lcb),
    )

# --------------------------
# Adversarial optimization for a single radius
# --------------------------
def adversarial_for_radius(dev, r, seed0):
    L = int(CFG["L"])
    beta = float(CFG["beta"])
    Ncand = int(CFG["Ncand"])
    steps = int(CFG["steps"])
    lr = float(CFG["lr"])
    grad_clip = float(CFG["grad_clip"])
    eps_fd = float(CFG["eps_fd"])
    mc_train = int(CFG["mc_train"])
    mc_val = int(CFG["mc_val"])
    mc_chunk = int(CFG["mc_chunk"])
    val_every = int(CFG["val_every"])
    z2_floor_opt = float(CFG["z2_floor_opt"])
    z2_penalty = float(CFG["z2_penalty"])
    clip_A_link = float(CFG["clip_A_link"])
    proj_iters = int(CFG["proj_iters"])
    proj_safety = float(CFG["proj_safety"])
    theta_slack = float(CFG["theta_slack"])
    sigma_init = float(CFG["sigma_init"])
    reinit_sigma = float(CFG["reinit_sigma"])
    reinit_absmax = float(CFG["reinit_absmax"])

    gen = torch.Generator(device=dev)
    gen.manual_seed(int(seed0))

    A = (sigma_init * torch.randn((Ncand, L, L, L, L, 4, 3), device=dev, dtype=torch.float32, generator=gen)).requires_grad_(True)
    opt = torch.optim.Adam([A], lr=lr, betas=(CFG["adam_beta1"], CFG["adam_beta2"]))

    with torch.no_grad():
        sanitize_and_reinit(A, gen, reinit_sigma, reinit_absmax)
        project_to_basin(A, r=r, clip_A_link=clip_A_link, iters=proj_iters, safety=proj_safety)

    best = dict(val=float("inf"))

    t0 = time.time()
    for it in range(steps):
        opt.zero_grad(set_to_none=True)

        sanitize_and_reinit(A, gen, reinit_sigma, reinit_absmax)

        U = su2_exp(A)
        z0flat, z2, zmax = compute_zflat_z2_zmax(U)

        P_train = P_hat_from_seeded_bank(U, z0flat, beta=beta, eps_fd=eps_fd,
                                         mc=mc_train, mc_chunk=mc_chunk, seed_bank=seed0 + 100000)
        ratio_train = P_train.to(z2.dtype) / torch.clamp(z2, min=z2_floor_opt)

        loss = ratio_train.sum() + z2_penalty * (z2_floor_opt / (z2 + z2_floor_opt)).sum()
        loss.backward()
        torch.nn.utils.clip_grad_norm_([A], max_norm=grad_clip)
        opt.step()

        with torch.no_grad():
            sanitize_and_reinit(A, gen, reinit_sigma, reinit_absmax)
            project_to_basin(A, r=r, clip_A_link=clip_A_link, iters=proj_iters, safety=proj_safety)

        if (it % val_every == 0) or (it == steps - 1):
            with torch.no_grad():
                Uv = su2_exp(A)
                z0v, z2v, zmaxv = compute_zflat_z2_zmax(Uv)
                thv = theta_max_plaq(Uv)

                P_val = P_hat_from_seeded_bank(Uv, z0v, beta=beta, eps_fd=eps_fd,
                                               mc=mc_val, mc_chunk=mc_chunk, seed_bank=seed0 + 200000)
                ratio_val = P_val.to(z2v.dtype) / torch.clamp(z2v, min=z2_floor_opt)

                feasible = (thv <= r + theta_slack) & (z2v > 0) & torch.isfinite(ratio_val)
                rv = ratio_val.clone()
                rv[~feasible] = float("inf")

                j = int(torch.argmin(rv).item())
                val = float(rv[j].item())
                if val < best["val"]:
                    best = dict(
                        val=val,
                        it=it,
                        idx=j,
                        theta=float(thv[j].item()),
                        z2=float(z2v[j].item()),
                        zmax=float(zmaxv[j].item()),
                        A_best=A[j].detach().cpu().clone().numpy().astype(np.float32),
                    )

                th_min = float(thv.min().item())
                th_med = float(thv.median().item())
                th_max = float(thv.max().item())
                print(f"[r={r:.3g} it {it:4d}] best_val={best['val']:.6g} (theta={best.get('theta',float('nan')):.4f}) "
                      f"| theta(min/med/max)={th_min:.4f}/{th_med:.4f}/{th_max:.4f}")

    best["wall_s"] = time.time() - t0
    return best

# --------------------------
# Main sweep
# --------------------------
def main():
    dev = torch.device("cuda" if torch.cuda.is_available() else "cpu") if CFG["device"] == "auto" else torch.device(CFG["device"])
    print("[device]", dev, (torch.cuda.get_device_name(0) if dev.type=="cuda" else ""))

    L = int(CFG["L"])
    assert L % 2 == 0
    beta = float(CFG["beta"])
    eps_fd = float(CFG["eps_fd"])

    base_seed = int(CFG["seed"])
    r_list = list(map(float, CFG["r_list"]))
    nR = len(r_list)

    best_val_ratio_opt = np.full((nR,), np.nan, dtype=np.float64)
    best_theta_opt = np.full((nR,), np.nan, dtype=np.float64)
    audit_ratio = np.full((nR,), np.nan, dtype=np.float64)
    audit_ratio_lcb = np.full((nR,), np.nan, dtype=np.float64)
    audit_P_hat = np.full((nR,), np.nan, dtype=np.float64)
    audit_P_se = np.full((nR,), np.nan, dtype=np.float64)
    audit_Z2 = np.full((nR,), np.nan, dtype=np.float64)
    audit_zmax = np.full((nR,), np.nan, dtype=np.float64)
    audit_theta = np.full((nR,), np.nan, dtype=np.float64)
    wall_s = np.full((nR,), np.nan, dtype=np.float64)

    for i, r in enumerate(r_list):
        print(f"\n=== RADIUS r={r} ===")
        best = adversarial_for_radius(dev, r=r, seed0=base_seed + 7777*i)

        A_best = torch.tensor(best["A_best"], device=dev, dtype=torch.float32).unsqueeze(0)
        with torch.no_grad():
            project_to_basin(A_best, r=r, clip_A_link=float(CFG["clip_A_link"]),
                             iters=int(CFG["proj_iters"]), safety=float(CFG["proj_safety"]))
            Ubest = su2_exp(A_best).to(torch.float64)

        ev = evaluate_high_mc(
            Ubest,
            beta=beta,
            eps_fd=eps_fd,
            mc=int(CFG["mc_eval"]),
            mc_chunk=int(CFG["mc_chunk_eval"]),
            seed=base_seed + 424242 + i,
            z2_min=float(CFG["z2_min_eval"]),
            nsigma=float(CFG["nsigma_eval"]),
        )

        print(f"[AUDIT r={r}] ratio_mean={ev['ratio']:.6g}  ratio_LCB({CFG['nsigma_eval']:.1f}σ)={ev['ratio_lcb']:.6g}  "
              f"theta_max={ev['theta_max']:.6g}  zmax={ev['zmax']:.6g}  Z2={ev['Z2']:.3e}")

        best_val_ratio_opt[i] = best["val"]
        best_theta_opt[i] = best["theta"]
        audit_ratio[i] = ev["ratio"]
        audit_ratio_lcb[i] = ev["ratio_lcb"]
        audit_P_hat[i] = ev["P_hat"]
        audit_P_se[i] = ev["P_se"]
        audit_Z2[i] = ev["Z2"]
        audit_zmax[i] = ev["zmax"]
        audit_theta[i] = ev["theta_max"]
        wall_s[i] = best["wall_s"]

    print("\n=== SUMMARY (high-MC audits) ===")
    for i, r in enumerate(r_list):
        print(f"r={r:<5}  ratio_LCB={audit_ratio_lcb[i]:.6g}  ratio_mean={audit_ratio[i]:.6g}  theta_max={audit_theta[i]:.6g}  wall={wall_s[i]/60:.2f} min")

    out = CFG.get("out_npz", None)
    if out:
        np.savez_compressed(
            out,
            CFG_text=str(CFG),
            r_list=np.array(r_list, dtype=np.float64),
            best_val_ratio_opt=best_val_ratio_opt,
            best_theta_opt=best_theta_opt,
            audit_ratio=audit_ratio,
            audit_ratio_lcb=audit_ratio_lcb,
            audit_P_hat=audit_P_hat,
            audit_P_se=audit_P_se,
            audit_Z2=audit_Z2,
            audit_zmax=audit_zmax,
            audit_theta=audit_theta,
            wall_s=wall_s,
        )
        print(f"[saved] {out}")

if __name__ == "__main__":
    main()


[device] cuda NVIDIA A100-SXM4-80GB

=== RADIUS r=0.1 ===
[r=0.1 it    0] best_val=72.2619 (theta=0.0985) | theta(min/med/max)=0.0985/0.0985/0.0986
[r=0.1 it   10] best_val=69.3644 (theta=0.0985) | theta(min/med/max)=0.0985/0.0985/0.0985
[r=0.1 it   20] best_val=69.3644 (theta=0.0985) | theta(min/med/max)=0.0985/0.0985/0.0985
[r=0.1 it   30] best_val=61.8502 (theta=0.0985) | theta(min/med/max)=0.0985/0.0985/0.0985
[r=0.1 it   40] best_val=61.8502 (theta=0.0985) | theta(min/med/max)=0.0985/0.0985/0.0985
[r=0.1 it   50] best_val=61.8502 (theta=0.0985) | theta(min/med/max)=0.0985/0.0985/0.0985
[r=0.1 it   60] best_val=61.8502 (theta=0.0985) | theta(min/med/max)=0.0985/0.0985/0.0989
[r=0.1 it   70] best_val=61.8502 (theta=0.0985) | theta(min/med/max)=0.0985/0.0985/0.0985
[r=0.1 it   80] best_val=50.5561 (theta=0.0985) | theta(min/med/max)=0.0985/0.0985/0.0985
[r=0.1 it   90] best_val=50.5561 (theta=0.0985) | theta(min/med/max)=0.0985/0.0985/0.0985
[r=0.1 it  100] best_val=50.5561 (theta=0.

In [ ]:
# Refined v4_final Configuration for L=12
CFG_L12_v4 = dict(
    device="cuda",
    L=12,                   # Targeted volume
    beta=6.0,               # Inverse coupling [cite: 64, 248]
    radii=[0.25],           # Focus on the most stable basin [cite: 68, 1160]
    Ncand=16,               # Parallel adversarial candidates [cite: 375, 1161]
    steps=250,              # Increased steps for L=12 convergence [cite: 376, 1168]
    lr=8e-3,                # Refined learning rate for stability [cite: 377, 1168]

    # Optimization Safeguards
    mc_bank=256,            # Larger frozen bank for smoother L=12 gradients [cite: 81, 1171]
    mc_chunk_bank=16,       # Memory management for A100 [cite: 82, 1172]
    eps_fd=1e-4,            # Finite difference epsilon [cite: 80, 1170]

    # High-Precision Audit (v4_final Logic)
    mc_audit=4096,          # 4096 fresh directions to prevent numerical noise [cite: 1174, 3358]
    mc_chunk_audit=64,      # Batching for high throughput [cite: 1175, 3359]
    nsigma_eval=2.0,        # Standard 2-sigma LCB [cite: 86, 1450]

    # Basin Constraints
    clip_A_link=0.40,       # Tightened link clamp for L=12 stability [cite: 836, 1178]
    proj_iters=12,          # Increased projection iterations for volume [cite: 836, 1178]
    z2_min=1e-6,            # Higher floor to block denominator exploits [cite: 87, 1182]

    seed=1234,              # Consistency with previous sweeps [cite: 89, 1183]
    out_npz="quartic_L12_v4_scaling_audit.npz"
)